In [ ]:
#pip install transformers sentencepiece torch sacremoses

In [14]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [24]:
import pyreadr
import pandas as pd
import numpy as np

In [34]:
# Wallenbeck data
df_w = pd.read_csv('data/wallenbeck_event_table_v3.csv')
print(df_w.columns.tolist())
print(df_w.head(3).to_string())

['Country', 'Dimension', 'Year Adopted', 'Left-Censored (pre-1991)', 'Pre-Directive', 'EU Deadline', 'Source', 'Confidence', 'Notes', 'gold_plating_category', 'years_after_sweden', 'in_corpus_window', 'lag_reliability']
  Country                              Dimension  Year Adopted Left-Censored (pre-1991) Pre-Directive  EU Deadline Source Confidence                                                                                                                                     Notes    gold_plating_category  years_after_sweden  in_corpus_window       lag_reliability
0     AUT                      Gestation Housing          2012                       NO           YES         2013   Text       HIGH                     Slightly more space than EU minimum since 2012 (survey data). Pre-directive for the 2012 addition (EU deadline 2013).            PRE_DIRECTIVE                24.0              True              RELIABLE
1     AUT  gestation_housing (confinement limit)          2018      

In [35]:
print(df_w.columns.tolist())
print(df_w.shape)
print(df_w.to_string())

['Country', 'Dimension', 'Year Adopted', 'Left-Censored (pre-1991)', 'Pre-Directive', 'EU Deadline', 'Source', 'Confidence', 'Notes', 'gold_plating_category', 'years_after_sweden', 'in_corpus_window', 'lag_reliability']
(22, 13)
   Country                              Dimension  Year Adopted Left-Censored (pre-1991) Pre-Directive  EU Deadline     Source Confidence                                                                                                                                                                                                        Notes           gold_plating_category  years_after_sweden  in_corpus_window                      lag_reliability
0      AUT                      Gestation Housing          2012                       NO           YES         2013       Text       HIGH                                                                                        Slightly more space than EU minimum since 2012 (survey data). Pre-directive for the 2012 additio

One small thing to note: DNK Tail Docking 2003 is classified as POST_DIRECTIVE_EXCEEDS_MINIMUM because the EU directive restricted routine docking in 1994 and Denmark's 2003 provision came after that. That is technically correct but worth flagging in the dissertation — Denmark's provision is qualitatively stricter than the EU's, not just a later adoption of the same standard. The lag reliability flag UNRELIABLE_LAG_POST_DIRECTIVE captures this adequately.

In [36]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Plotting config ───────────────────────────────────────────────────────────

CATEGORY_COLOURS = {
    'LEFT_CENSORED':                '#4393C3',   # blue   — Sweden reference
    'BEYOND_NO_EU_EQUIVALENT':      '#D6604D',   # red    — strongest voluntary
    'PRE_DIRECTIVE':                '#74C476',   # green  — reliable follower
    'SAME_YEAR_EXCEEDS_MINIMUM':    '#FD8D3C',   # orange — ambiguous
    'POST_DIRECTIVE_EXCEEDS_MINIMUM':'#BDBDBD',  # grey   — weakest signal
}

CATEGORY_LABELS = {
    'LEFT_CENSORED':                'Sweden (left-censored, pre-1991)',
    'BEYOND_NO_EU_EQUIVALENT':      'Beyond EU equivalent (no directive)',
    'PRE_DIRECTIVE':                'Pre-directive voluntary adoption',
    'SAME_YEAR_EXCEEDS_MINIMUM':    'Exceeds minimum, timed to directive',
    'POST_DIRECTIVE_EXCEEDS_MINIMUM':'Post-directive, exceeds minimum',
}

# EU directive deadlines — one horizontal line per dimension where relevant
EU_DEADLINES = {
    'Gestation Housing':    2013,
    'Lactation Housing':    2006,
    'Growing Finishing':    1998,
    'Weaning Age':          1994,
    'Tail Docking':         1994,
    'Manipulable Material': 2013,
}

# Country display order
COUNTRY_ORDER = ['SWE', 'AUT', 'DEU', 'DNK', 'FIN', 'NLD']

# Dimension display order — clean names
DIMENSION_ORDER = [
    'Gestation Housing',
    'Lactation Housing', 
    'Growing Finishing',
    'Weaning Age',
    'Tail Docking',
    'Manipulable Material',
]

# Normalise dimension names for matching
def normalise_dim(dim):
    """Map raw dimension strings to display names."""
    dim = str(dim).strip()
    mapping = {
        'Gestation Housing':                    'Gestation Housing',
        'gestation_housing (confinement limit)':'Gestation Housing',
        'Lactation Housing':                    'Lactation Housing',
        'Growing Finishing':                    'Growing Finishing',
        'Weaning Age':                          'Weaning Age',
        'Tail Docking':                         'Tail Docking',
        'Manipulable Material':                 'Manipulable Material',
    }
    return mapping.get(dim, dim)

df_plot = df_w.copy()
df_plot['dim_display'] = df_plot['Dimension'].apply(normalise_dim)

# ── Build figure ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(
    nrows=len(DIMENSION_ORDER),
    ncols=1,
    figsize=(12, 14),
    sharex=True
)

fig.suptitle(
    'Voluntary Pig Welfare Adoption Sequencing Relative to Sweden\n'
    'Six EU Voluntary-Adopter Countries, 1980–2020',
    fontsize=13, fontweight='bold', y=1.01
)

for ax, dim in zip(axes, DIMENSION_ORDER):
    
    subset = df_plot[df_plot['dim_display'] == dim].copy()
    
    # Draw EU directive deadline if applicable
    if dim in EU_DEADLINES:
        deadline = EU_DEADLINES[dim]
        ax.axvline(
            x=deadline, color='#525252', linewidth=1.2,
            linestyle='--', alpha=0.7, label='EU directive deadline'
        )
        ax.text(
            deadline + 0.3, len(COUNTRY_ORDER) - 0.1,
            f'EU {deadline}', fontsize=7, color='#525252', va='top'
        )
    
    # Plot each country's adoption event
    for y_pos, country in enumerate(COUNTRY_ORDER):
        country_rows = subset[subset['Country'] == country]
        
        if country_rows.empty:
            # No adoption — mark with a cross
            ax.plot(
                2020, y_pos, marker='x', color='#BDBDBD',
                markersize=6, markeredgewidth=1.5, alpha=0.5
            )
            continue
        
        for _, row in country_rows.iterrows():
            colour = CATEGORY_COLOURS.get(
                row['gold_plating_category'], '#BDBDBD'
            )
            # Corpus window indicator
            edgecolour = 'black' if row['in_corpus_window'] else 'none'
            
            ax.scatter(
                row['Year Adopted'], y_pos,
                color=colour, s=120,
                edgecolors=edgecolour, linewidths=1.5,
                zorder=3
            )
            
            # Annotate year
            ax.text(
                row['Year Adopted'], y_pos + 0.25,
                str(int(row['Year Adopted'])),
                fontsize=7, ha='center', color='#252525'
            )
            
            # Annotate lag for reliable cases
            if (row['lag_reliability'] == 'RELIABLE' and 
                row['Country'] != 'SWE' and
                not np.isnan(row['years_after_sweden'])):
                ax.text(
                    row['Year Adopted'], y_pos - 0.32,
                    f"+{int(row['years_after_sweden'])}y",
                    fontsize=6.5, ha='center', color='#525252', style='italic'
                )
    
    # Shade corpus window periods per country
    corpus_windows = {
        'AUT': (1996, 2019),
        'DEU': (2009, 2021),
        'DNK': (2007, 2022),
    }
    for y_pos, country in enumerate(COUNTRY_ORDER):
        if country in corpus_windows:
            start, end = corpus_windows[country]
            ax.barh(
                y_pos, end - start, left=start, height=0.6,
                color='#F7FBFF', alpha=0.4, zorder=0
            )
    
    # Axis formatting
    ax.set_yticks(range(len(COUNTRY_ORDER)))
    ax.set_yticklabels(COUNTRY_ORDER, fontsize=9)
    ax.set_ylabel(dim, fontsize=9, fontweight='bold', rotation=0,
                  labelpad=120, va='center')
    ax.set_xlim(1980, 2022)
    ax.grid(axis='x', linestyle=':', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('Year', fontsize=10)

# ── Legend ────────────────────────────────────────────────────────────────────

legend_elements = [
    mpatches.Patch(color=CATEGORY_COLOURS[cat], label=CATEGORY_LABELS[cat])
    for cat in CATEGORY_COLOURS
]
legend_elements.append(
    plt.Line2D([0], [0], linestyle='--', color='#525252',
               label='EU directive deadline')
)
legend_elements.append(
    plt.scatter([], [], s=80, edgecolors='black', facecolors='none',
                linewidths=1.5, label='Within ParlLawSpeech corpus window')
)

fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=3,
    fontsize=8,
    frameon=True,
    bbox_to_anchor=(0.5, -0.06)
)

plt.tight_layout()
plt.savefig(
    'adoption_sequencing_v1.png',
    dpi=180, bbox_inches='tight'
)
plt.show()
print("Saved adoption_sequencing_v1.png")

ModuleNotFoundError: No module named 'matplotlib'

### 2. ParlLawSpeech

To import the data, I start by creating a dictionaries for the countries and a list for the different parlimentary corpora.

In [12]:
COUNTRIES = {
    'austria': 'AT',
    'germany': 'DE',
    'denmark': 'DK'
}

DOC_TYPES = ['laws', 'bills', 'speeches']

The result of `pyreadr.read_r` is a dictionaries, whose keys are the name of objects, and the values python objects. In the case of RDS there is only one object, with `None` as key.

All the possible dataframes are a dictionary, whose keys are the differnet document types. For each document type there are three dictionaries, one for each country type. Through changing both these keys you can plug in the desired dataframe or loop through them again as used here to create them.

In [18]:
# Load all corpora into a nested dictionary: dfs[doc_type][country_code]
dfs = {}
for doc_type in DOC_TYPES:
    dfs[doc_type] = {}
    for country, code in COUNTRIES.items():
        path = f'data/Corpora_PLS_{country}/Corpus_{doc_type}_{country}.RDS'
        dfs[doc_type][code] = pyreadr.read_r(path)[None]

        print(f"Loaded {doc_type} [{code}]. (columns, rows): {dfs[doc_type][code].shape}")

Loaded laws [AT]: (columns, rows): (3030, 7)
Loaded laws [DE]: (columns, rows): (1638, 6)
Loaded laws [DK]: (columns, rows): (3220, 7)
Loaded bills [AT]: (columns, rows): (5926, 11)
Loaded bills [DE]: (columns, rows): (2445, 10)
Loaded bills [DK]: (columns, rows): (3615, 10)
Loaded speeches [AT]: (columns, rows): (204881, 14)
Loaded speeches [DE]: (columns, rows): (191932, 13)
Loaded speeches [DK]: (columns, rows): (716807, 11)


before we continue I need you to be aware of the format and layout of the parlawspeech file. It's stored as RDS files, where there are separate RDS files for the laws, speeches, and bills of a country during the period, and each country has their RDS files on those three things in their own folders. I can use pyreadr to convert these to dictionaries, and from there to dataframes.



The column structure for the different data types is as follows:





In [22]:
for doc_type in DOC_TYPES:
    print(f"\n{'='*60}")
    print(f"DOC TYPE: {doc_type.upper()}")
    print(f"{'='*60}")
    
    # Get all column sets
    col_sets = {code: set(dfs[doc_type][code].columns) 
                for code in COUNTRIES.values()}
    
    # Print columns per country
    for code, cols in col_sets.items():
        print(f"\n  [{code}] columns: {sorted(cols)}")
    
    # Find union and identify discrepancies
    all_cols = set.union(*col_sets.values())
    common_cols = set.intersection(*col_sets.values())
    
    print(f"\n  Common to all: {sorted(common_cols)}")
    print(f"  Not in all:    {sorted(all_cols - common_cols)}")
    
    # Show which country is missing which column
    for col in sorted(all_cols - common_cols):
        missing_in = [code for code, cols in col_sets.items() 
                      if col not in cols]
        present_in = [code for code, cols in col_sets.items() 
                      if col in cols]
        print(f"    '{col}': present in {present_in}, missing in {missing_in}")


DOC TYPE: LAWS

  [AT] columns: ['adoption_date', 'law_ID', 'law_link', 'law_text', 'period', 'procedure_ID', 'title_law']

  [DE] columns: ['adoption_date', 'law_ID', 'law_link', 'law_text', 'procedure_ID', 'title_law']

  [DK] columns: ['adoption_date', 'law_ID', 'law_link', 'law_text', 'period', 'procedure_ID', 'title_law']

  Common to all: ['adoption_date', 'law_ID', 'law_link', 'law_text', 'procedure_ID', 'title_law']
  Not in all:    ['period']
    'period': present in ['AT', 'DK'], missing in ['DE']

DOC TYPE: BILLS

  [AT] columns: ['bill_ID', 'bill_link', 'bill_text', 'bill_type', 'committee', 'initiation_date', 'initiator', 'period', 'procedure_ID', 'status', 'title_bill']

  [DE] columns: ['bill_ID', 'bill_link', 'bill_text', 'committee', 'initiation_date', 'initiator', 'period', 'procedure_ID', 'status', 'title_bill']

  [DK] columns: ['bill_ID', 'bill_link', 'bill_text', 'committee', 'initiation_date', 'initiator', 'period', 'procedure_ID', 'status', 'title_bill']

  Com

In [21]:
dfs['bills']['AT'].columns
dfs['speeches']['AT'].columns
dfs['laws']['AT'].columns

Index(['title_law', 'law_text', 'adoption_date', 'procedure_ID', 'law_link',
       'law_ID', 'period'],
      dtype='str')

In [13]:
bill_dfde['bill_text'].iloc[0]

'[Deutscher Bundestag Drucksache 19/23116 19. Wahlperiode 06.10.2020 Gesetzentwurf\\nder Abgeordneten Canan Bayram, Christian Kühn (Tübingen), Tabea Rößner,\\nDaniela Wagner, Claudia Müller, Luise Amtsberg, Britta Haßelmann, Katja Keul,\\nSven-Christian Kindler, Monika Lazar, Dr. Irene Mihalic, Dr. Konstantin von Notz,\\nLisa Paus, Filiz Polat, Dr. Manuela Rottmann, Stefan Schmidt, Markus Tressel,\\nDr. Julia Verlinden und der Fraktion BÜNDNIS 90/DIE GRÜNEN\\nEntwurf eines Gesetzes zur Ergänzung mietrechtlicher und\\ngewerbemietrechtlicher Regelungen des Bürgerlichen Gesetzbuches\\n– Mietrechts- und Gewerbemietrechtsergänzungsgesetz – A. Problem Gewerbemieterinnen und -mieter (es sind selbstverständlich stets alle\\nGeschlechter gemeint und angesprochen) stehen unter Druck. Zum einen führen langjährige\\nGentrifizierungsprozesse nicht nur mit Blick auf Wohnraummieterinnen und -mieter, sondern auch bei Gewerbemieterinnen und -mietern, etwa dem\\nkleinen inhabergeführten Einzelhandel, Ha

In [5]:
bill_df

,title_bill,bill_text,status,procedure_ID,initiation_date,bill_ID,initiator,period,bill_type,bill_link,committee
0,Ölkesseleinbauverbotsgesetz – ÖKEVG 2019,965/A XXVI. GP\r\n\r\nEingebracht am 02.07.201...,"Beschlossen im Bundesrat 240/BNR, Einhellig (Z...",26_965/A,2019-07-02,965/A,"Elisabeth Köstinger, MMMag. Dr. Axel Kassegger...",26,A,https://www.parlament.gv.at/gegenstand/XXVI/A/965,Wirtschaftsausschuss des Bundesrates
1,"Schulunterrichtsgesetz, Änderung",1023/A XXVI. GP\r\n\r\nEingebracht am 19.09.20...,"Zugewiesen an: Unterrichtsausschuss, Beratunge...",26_1023/A,2019-09-19,1023/A,Wendelin Mölzer,26,A,https://www.parlament.gv.at/gegenstand/XXVI/A/...,Unterrichtsausschuss
2,"Pensionsgesetz, Änderung",1021/A XXVI. GP\r\n\r\nEingebracht am 19.09.20...,"Zugewiesen an: Verfassungsausschuss, Beratunge...",26_1021/A,2019-09-19,1021/A,Werner Herbert,26,A,https://www.parlament.gv.at/gegenstand/XXVI/A/...,Verfassungsausschuss
3,"Gleichbehandlungsgesetz, Gesetz über die Gleic...",1020/A XXVI. GP\r\n\r\nEingebracht am 19.09.20...,"Zugewiesen an: Gleichbehandlungsausschuss, Ber...",26_1020/A,2019-09-19,1020/A,Gabriele Heinisch-Hosek,26,A,https://www.parlament.gv.at/gegenstand/XXVI/A/...,Gleichbehandlungsausschuss
4,"Kinderbetreuungsgeldgesetz, Änderung",1019/A XXVI. GP\r\n\r\nEingebracht am 19.09.20...,Zugewiesen an: Ausschuss für Familie und Jugen...,26_1019/A,2019-09-19,1019/A,Birgit Silvia Sandler,26,A,https://www.parlament.gv.at/gegenstand/XXVI/A/...,Ausschuss für Familie und Jugend
...,...,...,...,...,...,...,...,...,...,...,...
5921,2. BIG-Gesetz-Novelle,22 der Beilagen zu den Stenographischen Protok...,"Bekanntgabe des Gesetzesbeschlusses, Gesetzesv...",20_22 d.B.,1996-01-15,22 d.B.,Regierung,20,RV,https://www.parlament.gv.at/gegenstand/XX/I/22,Bautenausschuß
5922,Europa-Wählerevidenzgesetz,19 der Beilagen zu den Stenographischen Protok...,"Kein Einspruch, Gesetzesvorschlag in dritter L...",20_19 d.B.,1996-01-15,19 d.B.,Regierung,20,RV,https://www.parlament.gv.at/gegenstand/XX/I/19,Ausschuss für Verfassung und Föderalismus des ...
5923,Europawahlordnung - EuWO,18 der Beilagen zu den Stenographischen Protok...,"Kein Einspruch, Gesetzesvorschlag in dritter L...",20_18 d.B.,1996-01-15,18 d.B.,Regierung,20,RV,https://www.parlament.gv.at/gegenstand/XX/I/18,Ausschuss für Verfassung und Föderalismus des ...
5924,"Rechnungshofgesetz 1948, Änderung",16 der Beilagen zu den Stenographischen Protok...,"Kein Einspruch, Gesetzesvorschlag in dritter L...",20_16 d.B.,1996-01-15,16 d.B.,Regierung,20,RV,https://www.parlament.gv.at/gegenstand/XX/I/16,Ausschuss für Verfassung und Föderalismus des ...


In [6]:
law_df

,title_law,law_text,adoption_date,procedure_ID,law_link,law_ID,period
rownames,,,,,,,
532,Ölkesseleinbauverbotsgesetz – ÖKEVG 2019,BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r...,2020-01-20,26_965/A,https://www.ris.bka.gv.at/Dokumente/BgblAuth/B...,BGBl. I Nr. 6/2020,26
538,Gewaltschutzgesetz 2019,BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r...,2019-10-29,26_970/A,https://www.ris.bka.gv.at/Dokumente/BgblAuth/B...,BGBl. I Nr. 105/2019,26
539,Finanz-Organisationsreformgesetz – FORG,BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r...,2019-10-29,26_985/A,https://www.ris.bka.gv.at/Dokumente/BgblAuth/B...,BGBl. I Nr. 104/2019,26
540,Steuerreformgesetz 2020 – StRefG 2020,BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r...,2019-10-29,26_984/A,https://www.ris.bka.gv.at/Dokumente/BgblAuth/B...,BGBl. I Nr. 103/2019,26
541,Wehrrechtsänderungsgesetz 2019 – WRÄG 2019,BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r...,2019-10-25,26_509 d.B.,https://www.ris.bka.gv.at/Dokumente/BgblAuth/B...,BGBl. I Nr. 102/2019,26
...,...,...,...,...,...,...,...
10881,151. Bundesgesetz: Urheberrechtsgesetz-Novelle...,Der Nationalrat hat beschlossen: \r\nArtikel I...,1996-03-29,20_3 d.B.,https://www.ris.bka.gv.at/Dokumente/BgblPdf/19...,BGBl. Nr. 151/1996,20
10891,120. Bundesgesetz: 2. BIG-Gesetz-Novelle,Der Nationalrat hat beschlossen: \r\nDas BIG-G...,1996-03-14,20_22 d.B.,https://www.ris.bka.gv.at/Dokumente/BgblPdf/19...,BGBl. Nr. 120/1996,20
10901,119. Bundesgesetz: Änderung des Rechnungshofge...,Der Nationalrat hat beschlossen: \r\nDas Rechn...,1996-03-14,20_16 d.B.,https://www.ris.bka.gv.at/Dokumente/BgblPdf/19...,BGBl. Nr. 119/1996,20


Okay, my current data has the following columns for bills: title_bill	bill_text	status	procedure_ID	initiation_date	bill_ID	initiator	period	bill_type	bill_link	committee



and for laws: title_law	law_text	adoption_date	procedure_ID	law_link	law_ID	period



I do not yet see explanatory memoranda yet

In [7]:
law_df['law_text'].iloc[0]

'BUNDESGESETZBLATTFÜR DIE REPUBLIK ÖSTERREICH\r\n      \r\n      Jahrgang 2020\r\n        Ausgegeben am 20.\xa0Jänner 2020\r\n        Teil I\r\n      6. Bundesgesetz:Ölkesseleinbauverbotsgesetz – ÖKEVG\xa02019(NR: GP XXVI IA 965/A S.\xa089. BR: AB 10261 S.\xa0900.)6. Bundesgesetz über die Unzulässigkeit der Aufstellung und des Einbaus von Heizkesseln von Zentralheizungsanlagen für flüssige fossile oder für feste fossile Brennstoffe in Neubauten (Ölkesseleinbauverbotsgesetz – ÖKEVG\xa02019)\r\n      Der Nationalrat hat beschlossen:\r\n      §\xa01.(Verfassungsbestimmung) Die Erlassung, Änderung und Aufhebung von Vorschriften, wie sie in diesem Bundesgesetz enthalten sind, sind auch in den Belangen Bundessache, hinsichtlich derer das B-VG etwas anderes bestimmt.\r\n      §\xa02. Die Aufstellung und\xa0der Einbau von\xa0Heizkesseln von Zentralheizungsanlagen für flüssige fossile oder für feste fossile Brennstoffe in\xa0neu errichteten\xa0Gebäuden\xa0sind\xa0unzulässig. Dies ist in den\xa0

In [8]:
bill_df['bill_text'].iloc[0]

'965/A XXVI. GP\r\n\r\nEingebracht am 02.07.2019 Dieser Text ist elektronisch textinterpretiert. Abweichungen vom Original sind möglich.\r\n\r\nAntrag\r\n\r\nder Abgeordneten Elisabeth Köstinger, MMMag. Dr. Axel Kassegger, Josef Schellhorn, Mag. Josef Lettenbichler, Bruno Rossmann\r\n\r\nbetreffend ein Bundesgesetz über die Unzulässigkeit der Aufstellung und des Einbaus von Heizkesseln von Zentralheizungsanlagen für flüssige fossile oder für feste fossile Brennstoffe in Neubauten (Ölkesseleinbauverbotsgesetz – ÖKEVG 2019)\r\n\r\n\xa0\r\n\r\nDer Nationalrat wolle beschließen:\r\n\r\nBundesgesetz über die Unzulässigkeit der Aufstellung und des Einbaus von Heizkesseln von Zentralheizungsanlagen für flüssige fossile oder für feste fossile Brennstoffe in Neubauten (Ölkesseleinbauverbotsgesetz – ÖKEVG 2019)\r\n\r\nDer Nationalrat hat beschlossen:\r\n\r\n„§\xa01. (Verfassungsbestimmung) Die Erlassung, Änderung und Aufhebung von Vorschriften, wie sie in diesem Bundesgesetz enthalten sind, sind